# Análisis exploratorio del minidataset

> [!NOTE]
> Antes de ejecutar este notebook, necesitas crar el archivo manifest.csv y el archivo de configuración dataset.json. Ambos archivos pueden ser generados utilizando el notebook **inventario.ipynb**.

In [ ]:
# Importar bibiotecas necesarias
from pathlib import Path				# Para manejar rutas de archivos y directorios
import numpy as np						# Para procesar mascaras con etiquetas numéricas
from PIL import Image					# Para procesar imágenes
import json								# Para procesar archivos JSON
import pandas as pd						# Para procesar archivos CSV
from matplotlib import pyplot as plt	# Para visualizar imágenes y máscaras
from matplotlib.patches import Patch	# Para crear leyendas personalizadas en los gráficos

In [ ]:
# Leer el archivo de configuración
PROYECTO = Path("..")
ruta_config = PROYECTO / "config" / "dataset.json"
with ruta_config.open("r", encoding="utf-8") as archivo:
	config = json.load(archivo)

# Definir rutas de archivos y directorios
RUTA_MANIFEST = PROYECTO / config["manifest"]["path"]
DIRECTORIO_DATA = PROYECTO / config["dataset"]["root"]
IMAGES = DIRECTORIO_DATA / config["dataset"]["images"]

# Cargar el archivo manifest
manifest = pd.read_csv(RUTA_MANIFEST)
manifest.head()

# Análisis de categorías

### 📊 Distribución de imágenes por especie y raza

Antes de analizar las características de las imágenes, necesitamos conocer **cómo se distribuyen las muestras entre las categorías del dataset**.

En este minidataset, cada fotografía pertenece a una raza (`breed`) y, a su vez, cada raza pertenece a una especie (`species`). Por lo tanto, las categorías presentan una estructura jerárquica:

**Especie → Raza → Imágenes**

Para explorar esta organización, construiremos una **gráfica de barras horizontales** que muestre el número de imágenes disponibles para cada combinación de especie y raza.

#### 🔎 ¿Cómo interpretar la gráfica?

- **Eje vertical:** muestra las categorías, identificadas mediante su especie y raza.
- **Eje horizontal:** representa el número de imágenes pertenecientes a cada categoría.
- **Longitud de las barras:** permite comparar visualmente la cantidad de muestras entre las diferentes razas.
- **Colores:** distinguen las especies. Los tonos verdes corresponden a los gatos y los tonos terracota a los perros.

#### 💡 ¿Por qué es importante este análisis?

La cantidad de imágenes disponibles por categoría es una característica fundamental del dataset.

Cuando algunas categorías tienen considerablemente más muestras que otras, hablamos de un **desbalance en la distribución de categorías**.

Este desbalance puede ser relevante en tareas de clasificación, ya que un modelo podría recibir más ejemplos de determinadas categorías durante el entrenamiento.

Sin embargo, debemos distinguir entre dos tipos de distribución:

- **Distribución de imágenes por categoría:** cuántas fotografías pertenecen a cada especie o raza.
- **Distribución de píxeles por clase:** cuántos píxeles corresponden al animal, al fondo o al contorno dentro de las máscaras.

Aunque ambas son importantes, representan fenómenos diferentes. En esta gráfica analizamos únicamente la primera.

> [!NOTE]
> Nuestro objetivo final es desarrollar un modelo de segmentación de imágenes. Por ello, conocer la cantidad de fotografías por raza nos ayuda a caracterizar el dataset, pero no es suficiente para determinar si las clases de segmentación están balanceadas.

#### 🤔 Observa e interpreta

1. ¿Todas las razas tienen el mismo número de imágenes?
2. ¿Hay más fotografías de gatos o de perros?
3. ¿Qué información adicional necesitaríamos para determinar si las clases de segmentación están balanceadas?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

conteos = (
    manifest.groupby(["species", "breed"])
    .size()
    .sort_index()
)

etiquetas = [
    f"{especie} / {raza}"
    for especie, raza in conteos.index
]

colores_barras = [
    "#568A80" if especie == "cat" else "#D0805C"
    for especie, raza in conteos.index
]

barras = ax.barh(
    etiquetas,
    conteos.values,
    color=colores_barras
)

ax.bar_label(barras, padding=4)

ax.set_xlabel("Número de imágenes")
ax.set_ylabel("Especie / Raza")
ax.set_title("Distribución jerárquica de categorías")
ax.invert_yaxis()

ax.set_xlim(0, conteos.max() * 1.2)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 📊 Distribución proporcional de especies y razas

En la gráfica anterior comparamos el **número absoluto de imágenes** por categoría. Ahora analizaremos esa misma información desde una perspectiva diferente: **¿qué proporción del dataset representa cada categoría?**

Para ello, utilizaremos una **gráfica de anillos concéntricos (*nested donut chart*)**, que permite representar dos niveles de una estructura jerárquica.

En nuestro caso, los niveles son:

- **Especie:** gatos y perros.
- **Raza:** las categorías específicas que pertenecen a cada especie.

#### 🔎 ¿Cómo interpretar la gráfica?

La visualización está formada por dos anillos:

**Anillo interior — Especies**

Representa la proporción de imágenes que pertenece a cada especie respecto al total del minidataset.

Por ejemplo, un segmento que ocupe el 60 % del anillo indicaría que 6 de cada 10 imágenes pertenecen a esa especie.

**Anillo exterior — Razas**

Representa la proporción de imágenes correspondiente a cada raza respecto al total del minidataset.

Los colores permiten distinguir las razas, mientras que la leyenda indica el número absoluto de imágenes de cada una.

**Centro de la gráfica**

Muestra el número total de imágenes registradas en el manifest.

> [!IMPORTANT]
> Los porcentajes de ambos anillos se calculan respecto al **total de imágenes del dataset**.
>
> Por lo tanto, el porcentaje de una raza no representa su proporción dentro de su especie, sino su proporción respecto al conjunto completo.

#### 💡 ¿Qué aporta esta representación?

Aunque ambas gráficas utilizan los mismos conteos, permiten responder preguntas diferentes:

| Gráfica | Pregunta principal |
|---|---|
| Barras horizontales | ¿Cuántas imágenes tiene cada categoría? |
| Anillos concéntricos | ¿Qué proporción del dataset representa cada categoría? |

La representación proporcional facilita identificar qué especies y razas tienen mayor participación dentro del conjunto de datos.

Sin embargo, también tiene una limitación: **comparar ángulos y áreas suele ser menos preciso que comparar longitudes de barras**.

Por eso, estas visualizaciones son complementarias y no necesariamente sustitutas.

#### 🤔 Observa e interpreta

1. ¿Qué especie representa la mayor proporción del minidataset?
2. ¿La distribución de razas es uniforme?
3. ¿Una especie con más imágenes necesariamente tiene más razas representadas?
4. ¿Cuál de las dos gráficas utilizarías para comparar cantidades exactas y cuál para comunicar proporciones?

In [ ]:
# Contar imágenes por especie y raza
conteos = (
    manifest.groupby(["species", "breed"])
    .size()
    .sort_index()
)

# Totales por especie
especies = conteos.groupby(level="species").sum()

# Colores: tonos verdes para gatos y terracota para perros
colores_razas = {
    "Maine_Coon": "#568A80",
    "Siamese": "#78A080",
    "Sphynx": "#9FA757",
    "chihuahua": "#D0805C"
}

colores_especies = {
    "cat": "#2C6E63",
    "dog": "#B86846"
}

# Preparar datos
valores_razas = conteos.values

colores_exteriores = [
    colores_razas[raza]
    for especie, raza in conteos.index
]

colores_interiores = [
    colores_especies[especie]
    for especie in especies.index
]

# Crear figura
fig, ax = plt.subplots(figsize=(9, 7))

# Anillo exterior: razas
ax.pie(
    valores_razas,
    radius=1.0,
    colors=colores_exteriores,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.1f}%",
    pctdistance=0.85,
    wedgeprops={
        "width": 0.3,
        "edgecolor": "white",
        "linewidth": 2
    }
)

# Anillo interior: especies
ax.pie(
    especies.values,
    radius=0.68,
    colors=colores_interiores,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.1f}%",
    pctdistance=0.72,
    textprops={"color": "white", "weight": "bold"},
    wedgeprops={
        "width": 0.3,
        "edgecolor": "white",
        "linewidth": 2
    }
)

# Número total de imágenes en el centro
ax.text(
    0, 0,
    f"{len(manifest)}\nimágenes",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold"
)

# Leyenda de razas
leyenda = [
    Patch(
        facecolor=colores_razas[raza],
        label=f"{raza}: {cantidad} imágenes"
    )
    for (especie, raza), cantidad in conteos.items()
]

ax.legend(
    handles=leyenda,
    title="Razas",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "Distribución de imágenes por especie y raza",
    fontsize=14,
    fontweight="bold",
    pad=20
)

ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Análisis de Tamaño

### 📊 Distribución de las dimensiones de las imágenes

Hasta ahora hemos analizado cuántas imágenes pertenecen a cada categoría. Sin embargo, conocer la cantidad de muestras no nos dice nada sobre sus características espaciales.

Una imagen digital está formada por una cuadrícula de píxeles cuyas dimensiones se expresan mediante dos valores:

- **Ancho (`width`):** número de píxeles en la dirección horizontal.
- **Alto (`height`):** número de píxeles en la dirección vertical.

Por ejemplo, una fotografía de `600 × 400` tiene:

$$
N_{\text{píxeles}} = 600 \times 400 = 240\,000
$$

En esta actividad utilizaremos un **diagrama de dispersión (*scatter plot*)** para explorar la relación entre el ancho y el alto de las imágenes del minidataset.

#### 🔎 ¿Cómo interpretar la gráfica?

Cada punto representa una fotografía del dataset.

- **Eje horizontal (X):** ancho de la fotografía, expresado en píxeles.
- **Eje vertical (Y):** alto de la fotografía, expresado en píxeles.
- **Colores:** permiten distinguir las razas.
- **Línea diagonal de referencia:** representa las imágenes cuyo ancho y alto son iguales.

La posición de los puntos respecto a la diagonal permite identificar tres situaciones:

| Posición | Interpretación |
|---|---|
| Sobre la diagonal | La imagen es cuadrada: `width = height`. |
| Debajo de la diagonal | La imagen es más ancha que alta: `width > height`. |
| Encima de la diagonal | La imagen es más alta que ancha: `height > width`. |

También podemos observar si las fotografías se concentran alrededor de determinadas dimensiones o si existe una gran variabilidad de tamaños.

#### 💡 ¿Por qué es importante este análisis?

Durante el entrenamiento de una red neuronal, normalmente necesitamos organizar las imágenes en lotes (*batches*) representados mediante tensores de dimensiones compatibles.

Si las fotografías tienen tamaños diferentes, será necesario definir una estrategia de preparación de datos.

Algunas alternativas son:

- **Redimensionamiento (*resize*):** modificar las dimensiones de las imágenes.
- **Relleno (*padding*):** agregar píxeles alrededor de una imagen para alcanzar un tamaño determinado.
- **Recorte (*cropping*):** seleccionar una región de la imagen.

Cada estrategia tiene implicaciones diferentes.

Por ejemplo, redimensionar una fotografía sin conservar sus proporciones puede deformar los objetos, mientras que un recorte puede eliminar información importante.

> [!IMPORTANT]
> En segmentación, cualquier transformación geométrica aplicada a una fotografía debe aplicarse de manera consistente a su máscara.
>
> Además, al redimensionar máscaras con etiquetas discretas, debemos utilizar un método de interpolación apropiado, como **vecino más cercano**, para evitar generar valores de clase inexistentes.

#### 🤔 Observa e interpreta

1. ¿Todas las imágenes tienen las mismas dimensiones?
2. ¿Predominan las fotografías horizontales, verticales o cuadradas?
3. ¿Existen grupos de imágenes que comparten dimensiones similares?
4. ¿Observas diferencias en las dimensiones según la raza?
5. Si tuvieras que preparar estas imágenes para entrenar una U-Net, ¿qué estrategia utilizarías para unificar sus dimensiones?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for raza, grupo in manifest.groupby("breed"):

    ax.scatter(
        grupo["width"],
        grupo["height"],
        label=raza,
        alpha=0.7,
        s=50
    )

# Línea de referencia: ancho = alto
limite = max(
    manifest["width"].max(),
    manifest["height"].max()
)

ax.plot(
    [0, limite],
    [0, limite],
    linestyle="--",
    color="gray",
    label="Ancho = Alto"
)

ax.set_xlim(0, limite * 1.05)
ax.set_ylim(0, limite * 1.05)

ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("Ancho (píxeles)")
ax.set_ylabel("Alto (píxeles)")
ax.set_title("Dispersión de dimensiones por raza")

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 Distribución de la relación de aspecto

En la gráfica anterior analizamos las dimensiones de las fotografías mediante sus valores de ancho y alto.

Ahora estudiaremos una característica derivada de estas dimensiones: la **relación de aspecto (*aspect ratio*)**.

La relación de aspecto describe la proporción entre el ancho y el alto de una imagen:

$$
r = \frac{W}{H}
$$

Donde:

- $r$ es la relación de aspecto.
- $W$ es el ancho de la imagen.
- $H$ es el alto de la imagen.

Por ejemplo, una imagen de `600 × 400` píxeles tiene una relación de aspecto de:

$$
r = \frac{600}{400} = 1.5
$$

Esto significa que su ancho es 1.5 veces su alto.

#### 🔎 ¿Cómo interpretar los valores?

| Relación de aspecto | Interpretación |
|---|---|
| $r < 1$ | Imagen vertical: más alta que ancha. |
| $r = 1$ | Imagen cuadrada. |
| $r > 1$ | Imagen horizontal: más ancha que alta. |

A diferencia de las dimensiones absolutas, la relación de aspecto no depende directamente del número total de píxeles.

Por ejemplo, las imágenes de `600 × 400` y `1200 × 800` tienen diferentes dimensiones y cantidades de píxeles, pero comparten la misma relación de aspecto:

$$
r = 1.5
$$

#### 🔎 ¿Cómo interpretar el histograma?

El histograma muestra cómo se distribuyen las relaciones de aspecto de las fotografías.

- **Eje horizontal (X):** valores de la relación de aspecto.
- **Eje vertical (Y):** número de imágenes dentro de cada intervalo.
- **Barras:** indican cuántas fotografías presentan relaciones de aspecto similares.

A diferencia de una gráfica de barras por categoría, un histograma agrupa los valores numéricos en intervalos llamados *bins*.

Esto permite identificar si las imágenes presentan proporciones similares o si existe una amplia diversidad de formatos.

#### 💡 ¿Por qué es importante este análisis?

La relación de aspecto es especialmente relevante cuando necesitamos unificar las dimensiones de las imágenes.

Supongamos que queremos transformar todas las fotografías a un tamaño de `256 × 256` píxeles.

Si una fotografía original tiene dimensiones de `600 × 400`, su relación de aspecto es:

$$
r_{\text{original}} = 1.5
$$

Después de redimensionarla a `256 × 256`, tendremos:

$$
r_{\text{final}} = 1
$$

La fotografía habrá cambiado de proporciones y, por lo tanto, **los objetos pueden aparecer deformados**.

Una alternativa consiste en redimensionar conservando la relación de aspecto y completar las dimensiones restantes mediante *padding*.

> [!NOTE]
> No existe una única estrategia de redimensionamiento adecuada para todos los problemas.
>
> La elección depende de las características del dataset, los requisitos de la arquitectura y la importancia de preservar la geometría de los objetos.

#### 🤔 Observa e interpreta

1. ¿Qué intervalo de relaciones de aspecto concentra la mayor cantidad de fotografías?
2. ¿Predominan las imágenes horizontales o verticales?
3. ¿Existen imágenes con proporciones considerablemente diferentes al resto?
4. ¿Dos imágenes pueden tener la misma relación de aspecto y diferente número de píxeles?
5. ¿Qué consecuencias podría tener redimensionar todas las fotografías a un formato cuadrado sin preservar sus proporciones?

In [ ]:
# Calcular relación de aspecto
manifest["aspect_ratio"] = manifest["width"] / manifest["height"]

# Crear histograma
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    manifest["aspect_ratio"],
    bins=15,
    color="#568A80",
    edgecolor="white",
    alpha=0.85
)

# Línea de referencia para imágenes cuadradas
ax.axvline(
    x=1,
    color="#D0805C",
    linestyle="--",
    linewidth=2,
    label="Relación 1:1 (cuadrada)"
)

ax.set_xlabel("Relación de aspecto (ancho / alto)")
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución de relaciones de aspecto")

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(manifest["aspect_ratio"].describe().round(2))

## Análisis de máscaras

### 📊 Distribución del porcentaje de píxeles del animal

Hasta ahora hemos estudiado las características de las fotografías: sus categorías, dimensiones y relaciones de aspecto.

Sin embargo, en un problema de **segmentación semántica**, también necesitamos conocer cómo se distribuyen las regiones etiquetadas dentro de las máscaras.

Recordemos que las máscaras del Oxford-IIIT Pet Dataset utilizan tres valores:

| Valor | Región | Descripción |
|---|---|---|
| `1` | Animal | Píxeles correspondientes al animal (*foreground*). |
| `2` | Fondo | Píxeles correspondientes al fondo (*background*). |
| `3` | Contorno | Píxeles de la región de borde o de clasificación ambigua. |

Para cada máscara podemos contar cuántos píxeles pertenecen a cada una de estas regiones.

En particular, nos interesa conocer **qué proporción de la imagen está ocupada por el animal**.

#### 🔢 ¿Cómo calculamos el porcentaje de píxeles del animal?

Primero contamos el número de píxeles cuya etiqueta es `1`:

$$
N_{\text{animal}} = \sum_{i=1}^{H}\sum_{j=1}^{W} \mathbb{1}[M(i,j)=1]
$$

Donde:

- $M$ representa la máscara de segmentación.
- $H$ y $W$ son sus dimensiones.
- $\mathbb{1}[\cdot]$ es una función indicadora que vale `1` cuando se cumple la condición y `0` en caso contrario.

Después calculamos el porcentaje respecto al número total de píxeles:

$$
P_{\text{animal}} =
\frac{N_{\text{animal}}}{H \times W}\times 100
$$

Por ejemplo, supongamos que una máscara contiene `200 000` píxeles y que `80 000` pertenecen al animal:

$$
P_{\text{animal}} =
\frac{80\,000}{200\,000}\times 100 = 40\%
$$

Esto significa que el animal ocupa el **40 % de la superficie de la imagen**.

> [!NOTE]
> Este porcentaje describe la proporción de píxeles etiquetados como animal, no el tamaño físico real del animal.
>
> Una fotografía tomada de cerca puede mostrar un animal que ocupa gran parte de la imagen, mientras que una fotografía tomada desde lejos puede mostrar al mismo animal ocupando una región mucho menor.

---

#### 🔎 ¿Cómo interpretar el histograma?

El histograma representa la distribución del porcentaje de píxeles del animal entre las diferentes muestras del dataset.

- **Eje horizontal (X):** porcentaje de píxeles etiquetados como animal.
- **Eje vertical (Y):** número de imágenes dentro de cada intervalo.
- **Barras:** representan la frecuencia con la que aparecen determinados porcentajes.

Cada fotografía aporta un valor al histograma, calculado a partir de su máscara correspondiente.

Por ejemplo:

- Un valor cercano al **10 %** indica que el animal ocupa una pequeña proporción de la imagen.
- Un valor cercano al **50 %** indica que aproximadamente la mitad de los píxeles pertenecen al animal.
- Un valor cercano al **90 %** indica que el animal ocupa casi toda la imagen.

La forma del histograma nos permitirá identificar si las fotografías presentan proporciones similares o si existe una gran variabilidad en el espacio ocupado por los animales.

---

#### 💡 ¿Por qué es importante este análisis?

La proporción de píxeles del objeto es relevante para comprender las características del problema de segmentación.

**1. Variabilidad del tamaño relativo de los objetos**

Si los porcentajes varían considerablemente entre imágenes, significa que el modelo deberá aprender a identificar animales que ocupan proporciones diferentes del campo visual.

Esto puede representar un desafío adicional, especialmente cuando los objetos aparecen a escalas muy distintas.

**2. Desbalance entre clases de segmentación**

En segmentación, cada píxel representa una observación etiquetada.

Por lo tanto, podemos encontrar datasets donde una clase ocupa una proporción mucho mayor de píxeles que las demás.

Por ejemplo, si la mayoría de las máscaras contiene una gran cantidad de píxeles de fondo y relativamente pocos del animal, existe un desbalance entre las clases a nivel de píxel.

Este fenómeno puede influir en el entrenamiento, especialmente cuando utilizamos funciones de pérdida que consideran cada píxel de manera individual.

**3. Selección de métricas de evaluación**

Cuando las clases presentan distribuciones desiguales, una métrica como la exactitud global (*pixel accuracy*) puede resultar poco informativa.

Por ejemplo, si el fondo ocupa el 90 % de una imagen, un modelo que prediga únicamente fondo podría obtener una exactitud del 90 %, a pesar de no segmentar correctamente al animal.

Por esta razón, en segmentación suelen utilizarse métricas como:

- **Intersection over Union (IoU)**.
- **Dice Similarity Coefficient (DSC)**.

Estas métricas permiten evaluar el grado de coincidencia entre las regiones predichas y las regiones de referencia.

> [!IMPORTANT]
> **El porcentaje de píxeles del animal no es una medida directa del desempeño de un modelo.**
>
> Es una característica del dataset que nos ayuda a comprender la distribución espacial de las clases y anticipar posibles dificultades durante el entrenamiento.

---

#### 🤔 Observa e interpreta

1. ¿En qué intervalo se concentra la mayor cantidad de fotografías?
2. ¿Los animales ocupan proporciones similares en todas las imágenes?
3. ¿Existen fotografías donde el animal ocupa una proporción muy pequeña?
4. ¿Qué dificultades podría presentar un modelo al segmentar animales que ocupan una región pequeña de la imagen?
5. ¿Por qué una distribución desbalanceada de píxeles puede afectar la interpretación de la exactitud global?
6. ¿Podemos concluir que el dataset está balanceado únicamente observando este histograma? ¿Qué información adicional necesitaríamos?

In [ ]:
def analizar_mascara(ruta_mascara):

    with Image.open(ruta_mascara) as img:
        mascara = np.array(img)

    # Contar píxeles por clase
    pixeles_animal = np.count_nonzero(mascara == 1)
    pixeles_fondo = np.count_nonzero(mascara == 2)
    pixeles_contorno = np.count_nonzero(mascara == 3)

    pixeles_totales = mascara.size

    return {
        "pixels_total": pixeles_totales,
        "pixels_animal": pixeles_animal,
        "pixels_fondo": pixeles_fondo,
        "pixels_contorno": pixeles_contorno,
        "pct_animal": pixeles_animal / pixeles_totales * 100,
        "pct_fondo": pixeles_fondo / pixeles_totales * 100,
        "pct_contorno": pixeles_contorno / pixeles_totales * 100
    }

In [ ]:
# Analizar todas las máscaras y crear un DataFrame con los resultados
registros = []

for _, fila in manifest.iterrows():

    ruta_mascara = IMAGES / fila["mask_path"]

    datos = analizar_mascara(ruta_mascara)

    registros.append({
        "image_id": fila["image_id"],
        "breed": fila["breed"],
        "species": fila["species"],
        **datos
    })

df_mascaras = pd.DataFrame(registros)

display(df_mascaras.head())

In [ ]:
# Verificar consistencia de píxeles
df_mascaras["pixels_verificados"] = (
    df_mascaras["pixels_animal"]
    + df_mascaras["pixels_fondo"]
    + df_mascaras["pixels_contorno"]
)

print(
    "Máscaras con etiquetas inesperadas:",
    (df_mascaras["pixels_verificados"] != df_mascaras["pixels_total"]).sum()
)

In [ ]:
# Crear histograma del porcentaje de píxeles del animal
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    df_mascaras["pct_animal"],
    bins=12,
    color="#568A80",
    edgecolor="white",
    alpha=0.85
)

ax.axvline(
    df_mascaras["pct_animal"].median(),
    color="#D0805C",
    linestyle="--",
    linewidth=2,
    label=f"Mediana: {df_mascaras['pct_animal'].median():.1f}%"
)

ax.set_xlabel("Porcentaje de píxeles del animal (%)")
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución del área ocupada por el animal")

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 Composición porcentual de las máscaras de segmentación

En el histograma anterior analizamos qué porcentaje de cada imagen está ocupado por el animal. Sin embargo, las máscaras del Oxford-IIIT Pet Dataset contienen tres regiones: **animal, fondo y contorno**.

Ahora estudiaremos cómo se distribuyen estas tres regiones dentro de cada máscara mediante una **gráfica de barras apiladas al 100 % (*100% stacked bar chart*)**.

#### 🔎 ¿Cómo interpretar la gráfica?

Cada barra representa una imagen del dataset y está dividida en tres segmentos:

- **Animal:** porcentaje de píxeles etiquetados como animal.
- **Fondo:** porcentaje de píxeles correspondientes al fondo.
- **Contorno:** porcentaje de píxeles pertenecientes a la región de borde o clasificación ambigua.

La suma de los tres segmentos de cada barra es igual al **100 % de los píxeles de la máscara**.

Esta representación permite comparar la composición de diferentes imágenes, independientemente de sus dimensiones originales.

Por ejemplo, una barra con un segmento grande de animal indica que este ocupa una proporción considerable de la fotografía. En cambio, una barra dominada por el fondo indica que el animal ocupa una región relativamente pequeña.

#### 💡 ¿Qué información podemos obtener?

A diferencia del histograma anterior, esta gráfica permite observar **simultáneamente las tres clases de segmentación**.

Podemos identificar:

- Imágenes donde predomina el animal.
- Imágenes donde predomina el fondo.
- Diferencias en la proporción de píxeles de contorno.
- Muestras con composiciones considerablemente diferentes al resto.

También podemos observar si existe alguna relación entre la proporción del animal y la del fondo.

> [!NOTE]
> Aunque todas las barras tienen la misma altura, esto no significa que las imágenes tengan el mismo número de píxeles.
>
> La gráfica representa **porcentajes**, no cantidades absolutas. Una imagen pequeña y una grande pueden tener composiciones porcentuales similares, aunque contengan cantidades muy diferentes de píxeles.

#### 🤔 Observa e interpreta

1. ¿Qué región predomina en la mayoría de las máscaras?
2. ¿Todas las imágenes presentan proporciones similares de animal, fondo y contorno?
3. ¿Existen muestras donde el contorno representa una proporción considerable de la máscara?
4. ¿Podemos conocer el número total de píxeles de cada imagen únicamente a partir de esta gráfica?

In [ ]:
# Crear gráfico de barras apiladas del porcentaje de píxeles por clase
df_ordenado = df_mascaras.sort_values("pct_animal")

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_ordenado))

ax.bar(
    x,
    df_ordenado["pct_animal"],
    label="Animal",
    color="#568A80"
)

ax.bar(
    x,
    df_ordenado["pct_fondo"],
    bottom=df_ordenado["pct_animal"],
    label="Fondo",
    color="#93A3C5"
)

ax.bar(
    x,
    df_ordenado["pct_contorno"],
    bottom=(
        df_ordenado["pct_animal"]
        + df_ordenado["pct_fondo"]
    ),
    label="Contorno",
    color="#D0805C"
)

ax.set_xlabel("Imágenes ordenadas por área del animal")
ax.set_ylabel("Porcentaje de píxeles (%)")
ax.set_title("Distribución de clases en cada máscara")

ax.set_ylim(0, 100)
ax.set_xticks([])
ax.legend()

plt.tight_layout()
plt.show()

### 📊 Distribución global de píxeles por clase

Hasta ahora hemos analizado la composición de las máscaras de manera individual. Sin embargo, para caracterizar el dataset completo también necesitamos conocer **qué proporción del total de píxeles pertenece a cada clase de segmentación**.

Para ello, utilizaremos una **gráfica de pastel (*pie chart*)** que represente la distribución global de las tres clases: animal, fondo y contorno.

#### 🔎 ¿Cómo interpretar la gráfica?

Cada sector representa una clase de segmentación:

- **Animal:** proporción global de píxeles etiquetados como animal.
- **Fondo:** proporción global de píxeles etiquetados como fondo.
- **Contorno:** proporción global de píxeles correspondientes a la región de borde.

El tamaño de cada sector depende de la cantidad de píxeles de esa clase respecto al total de píxeles del dataset.

La suma de los porcentajes de los tres sectores es igual al **100 %**.

#### 🔢 ¿Cómo se calcula la proporción global?

Para una clase $c$, primero sumamos sus píxeles en todas las máscaras:

$$
N_c = \sum_{k=1}^{n} N_{c,k}
$$

Donde:

- $n$ es el número de máscaras.
- $N_{c,k}$ es la cantidad de píxeles de la clase $c$ en la máscara $k$.

Después calculamos su proporción respecto al total de píxeles:

$$
P_c = \frac{N_c}{N_{\text{total}}}\times 100
$$

Donde $N_{\text{total}}$ representa la suma de los píxeles de todas las máscaras.

#### 💡 ¿Por qué no basta con promediar los porcentajes por imagen?

Cuando las imágenes tienen dimensiones diferentes, **el promedio de los porcentajes individuales no necesariamente coincide con la proporción global de píxeles**.

Consideremos dos máscaras:

| Máscara | Píxeles totales | Píxeles del animal | Porcentaje del animal |
|---|---:|---:|---:|
| Imagen A | 100 | 80 | 80 % |
| Imagen B | 900 | 180 | 20 % |

Si calculamos el promedio de los porcentajes individuales:

$$
P_{\text{promedio}} = \frac{80+20}{2}=50\%
$$

Sin embargo, si sumamos todos los píxeles del animal y los dividimos entre el total de píxeles:

$$
P_{\text{global}} =
\frac{80+180}{100+900}\times100=26\%
$$

**¿Por qué existe esta diferencia?**

Porque el promedio de porcentajes asigna el mismo peso a cada imagen, independientemente de su tamaño.

En cambio, la proporción global considera todos los píxeles del dataset, por lo que las imágenes de mayor resolución contribuyen con una cantidad mayor.

> [!IMPORTANT]
> **Promedio de porcentajes por imagen ≠ Proporción global de píxeles**, cuando las imágenes tienen diferentes tamaños y composiciones.
>
> Ambas medidas son válidas, pero responden preguntas distintas:
>
> - El **promedio por imagen** describe la composición porcentual promedio de una muestra.
> - La **proporción global** describe cómo se distribuyen todos los píxeles del dataset entre las clases.

#### 💡 ¿Por qué es importante este análisis?

La distribución global permite identificar posibles **desbalances entre las clases de segmentación**.

Por ejemplo, si el fondo representa una proporción considerablemente mayor que el animal, el modelo podría estar expuesto a muchos más píxeles de fondo durante el entrenamiento.

Este fenómeno puede influir en el aprendizaje y motivar decisiones posteriores sobre:

- Funciones de pérdida.
- Estrategias de muestreo.
- Métricas de evaluación.
- Tratamiento de clases minoritarias.

> [!NOTE]
> La distribución global caracteriza el dataset analizado, pero no determina por sí sola el desempeño futuro de un modelo.
>
> Además, si posteriormente dividimos el dataset en entrenamiento, validación y prueba, será conveniente revisar estas distribuciones por separado.

#### 🤔 Observa e interpreta

1. ¿Qué clase representa la mayor proporción de píxeles del dataset?
2. ¿Existe un desbalance considerable entre animal, fondo y contorno?
3. ¿Por qué la proporción global puede diferir del promedio de los porcentajes individuales?
4. ¿Qué consecuencias podría tener un desbalance importante durante el entrenamiento de un modelo de segmentación?
5. ¿Qué información adicional necesitaríamos antes de decidir si debemos aplicar alguna estrategia para tratar el desbalance?

In [ ]:
# Crear gráfico de pastel con la distribución global de píxeles por clase
totales = df_mascaras[
    ["pixels_animal", "pixels_fondo", "pixels_contorno"]
].sum()

fig, ax = plt.subplots(figsize=(7, 6))

ax.pie(
    totales.values,
    labels=["Animal", "Fondo", "Contorno"],
    colors=["#568A80", "#93A3C5", "#D0805C"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2
    }
)

ax.set_title("Distribución global de píxeles por clase")

plt.tight_layout()
plt.show()

### 📊 Distribución de los pixeles del animal (sin considerar el contorno)

In [ ]:
# Crear gráfico de violín para el número de píxeles del animal
# Número de píxeles correspondientes al animal
datos = df_mascaras["pixels_animal"].dropna().to_numpy()

# Estadísticas descriptivas
media = np.mean(datos)
mediana = np.median(datos)
q1, q3 = np.percentile(datos, [25, 75])

# Crear figura
fig, ax = plt.subplots(figsize=(6, 7))

# Violin plot
violin = ax.violinplot(
    datos,
    positions=[1],
    widths=0.7,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Estilo del violín
for cuerpo in violin["bodies"]:
    cuerpo.set_facecolor("#568A80")
    cuerpo.set_edgecolor("#2C3431")
    cuerpo.set_alpha(0.45)

# Observaciones individuales
rng = np.random.default_rng(42)
x = 1 + rng.uniform(-0.12, 0.12, size=len(datos))

ax.scatter(
    x, datos,
    color="#2C3431",
    s=16,
    alpha=0.45,
    zorder=3,
    label="Imágenes individuales"
)

# Rango intercuartílico
ax.vlines(
    1, q1, q3,
    color="#2C3431",
    linewidth=7,
    alpha=0.85,
    zorder=4,
    label="Rango intercuartílico (Q1-Q3)"
)

# Cuartiles Q1 y Q3
ax.scatter(
    [1, 1],
    [q1, q3],
    color="#2C3431",
    s=45,
    zorder=5
)

# Mediana
ax.hlines(
    mediana,
    xmin=0.82,
    xmax=1.18,
    color="#CD972B",
    linewidth=3,
    zorder=5,
    label=f"Mediana: {mediana:,.0f} px"
)

# Media
ax.scatter(
    1, media,
    color="#D0805C",
    marker="D",
    s=85,
    edgecolor="white",
    zorder=6,
    label=f"Media: {media:,.0f} px"
)

# Configuración
ax.set_xticks([1])
ax.set_xticklabels(["Animal"])
ax.set_ylabel("Número de píxeles del animal")
ax.set_title("Distribución del tamaño absoluto del animal")

ax.set_xlim(0.5, 1.5)
ax.set_ylim(bottom=0)

ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

print(f"Número de máscaras: {len(datos)}")
print(f"Media:   {media:,.0f} píxeles")
print(f"Mediana: {mediana:,.0f} píxeles")
print(f"Q1:      {q1:,.0f} píxeles")
print(f"Q3:      {q3:,.0f} píxeles")
print(f"RIC:     {q3 - q1:,.0f} píxeles") # Rango intercuartílico (RIC)

### ¿Cómo interpretar un violin plot?

Un **violin plot (gráfico de violín)** permite visualizar la distribución de una variable numérica, combinando información sobre la concentración de los datos y sus principales estadísticas descriptivas.

En este caso, analizamos la **cantidad de píxeles etiquetados como animal** en cada máscara de segmentación, sin incluir los píxeles correspondientes al contorno.

**Elementos del gráfico:**

- **Ancho del violín:** representa la densidad estimada de los datos. Las regiones más anchas indican una mayor concentración de observaciones en ese intervalo; las regiones estrechas indican una menor concentración.
- **Puntos individuales:** representan las observaciones de cada máscara, permitiendo identificar la dispersión de los valores y posibles casos extremos.
- **Media (rombo terracota):** representa el promedio de píxeles etiquetados como animal.
- **Mediana (línea mostaza):** indica el valor central de la distribución; el 50 % de las observaciones se encuentra a cada lado de este valor.
- **Cuartiles Q1 y Q3 (extremos de la barra oscura):** delimitan el intervalo que contiene el 50 % central de las observaciones.
- **Rango intercuartílico (RIC = Q3 − Q1):** mide la dispersión del 50 % central de los datos.

**¿Qué debemos observar?**

1. **Concentración:** ¿en qué intervalo de píxeles se encuentran la mayoría de las máscaras?
2. **Variabilidad:** ¿los tamaños de las regiones segmentadas son similares o presentan diferencias importantes?
3. **Asimetría:** ¿la media y la mediana están próximas o existe una separación considerable?
4. **Valores extremos:** ¿hay máscaras con cantidades de píxeles mucho mayores o menores que las demás?

> **Importante:** el ancho del violín es una estimación suavizada de la densidad, no un conteo directo de imágenes. Además, el gráfico representa tamaños absolutos en píxeles, por lo que las diferencias observadas pueden estar relacionadas tanto con el área ocupada por el animal como con la resolución original de las imágenes. Para comparar proporciones independientemente de la resolución, conviene complementar este análisis con `pct_animal`.

In [ ]:
# Crear gráfico de violín para el porcentaje de píxeles del animal
# Porcentaje de píxeles correspondientes al animal (etiqueta 1)
datos = df_mascaras["pct_animal"].dropna().to_numpy()

# Estadísticas descriptivas
media = np.mean(datos)
mediana = np.median(datos)
q1, q3 = np.percentile(datos, [25, 75])

fig, ax = plt.subplots(figsize=(6, 7))

# Violin plot
ax.violinplot(
    datos,
    positions=[1],
    widths=0.7,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Ajustar color del violín
violin = ax.collections[0]
violin.set_facecolor("#568A80")
violin.set_edgecolor("#2C3431")
violin.set_alpha(0.45)

# Observaciones individuales (jitter reproducible)
rng = np.random.default_rng(42)
x = 1 + rng.uniform(-0.12, 0.12, size=len(datos))

ax.scatter(
    x, datos,
    color="#2C3431",
    s=16,
    alpha=0.45,
    zorder=3,
    label="Imágenes individuales"
)

# Rango intercuartílico Q1-Q3
ax.vlines(
    1, q1, q3,
    color="#2C3431",
    linewidth=7,
    alpha=0.85,
    zorder=4,
    label="Rango intercuartílico (Q1-Q3)"
)

# Media
ax.scatter(
    1, media,
    color="#D0805C",
    marker="D",
    s=85,
    edgecolor="white",
    zorder=6,
    label=f"Media: {media:.2f}%"
)

# Mediana
ax.hlines(
    mediana,
    xmin=0.82,
    xmax=1.18,
    color="#CD972B",
    linewidth=3,
    zorder=5,
    label=f"Mediana: {mediana:.2f}%"
)

# Cuartiles
ax.scatter(
    [1, 1],
    [q1, q3],
    color="#2C3431",
    s=45,
    zorder=5
)

ax.set_xticks([1])
ax.set_xticklabels(["Animal"])
ax.set_ylabel("Área ocupada por el animal (%)")
ax.set_title("Distribución del tamaño relativo del animal")

ax.set_xlim(0.5, 1.5)
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

print(f"Número de máscaras: {len(datos)}")
print(f"Media:   {media:.2f}%")
print(f"Mediana: {mediana:.2f}%")
print(f"Q1:      {q1:.2f}%")
print(f"Q3:      {q3:.2f}%")
print(f"RIC:     {q3 - q1:.2f} puntos porcentuales")

---

### 🧠 Integrando los resultados

Durante este análisis utilizamos diferentes representaciones para estudiar las máscaras de segmentación desde distintas perspectivas.

| Visualización | Unidad de análisis | Información principal |
|---|---|---|
| Histograma del porcentaje del animal | Cada imagen | Distribución del tamaño relativo del animal dentro de las fotografías. |
| Gráfica de violín del número de píxeles del animal | Cada imagen | Distribución de la cantidad absoluta de píxeles del animal, incluyendo dispersión, concentración, media, mediana y cuartiles. |
| Gráfica de violín del porcentaje de píxeles del animal | Cada imagen | Distribución del tamaño relativo del animal, independientemente de la resolución de cada fotografía. |
| Barras apiladas | Cada imagen | Composición porcentual de animal, fondo y contorno. |
| Gráfica de pastel | Dataset completo | Distribución global de píxeles entre las tres clases de segmentación. |

Aunque todas estas visualizaciones utilizan información de las mismas máscaras, **cada una responde una pregunta diferente**.

#### 🔎 Cantidades absolutas frente a proporciones

Las gráficas de violín nos permiten distinguir dos perspectivas:

- **Número de píxeles del animal:** indica cuántos píxeles están etiquetados como animal en cada máscara. Este valor depende tanto de la resolución de la imagen como del espacio ocupado por el objeto.
- **Porcentaje de píxeles del animal:** indica qué proporción de la imagen está ocupada por el animal, permitiendo comparar muestras con diferentes dimensiones.

Por ejemplo, una imagen de mayor resolución puede contener más píxeles del animal que otra, aunque el objeto ocupe una proporción menor de la fotografía.

Por eso, **una cantidad absoluta elevada no implica necesariamente una proporción elevada**.

#### 📊 Distribución, dispersión y composición

Las gráficas de violín también complementan al histograma porque permiten observar:

- **Concentración:** en qué intervalos se acumulan más observaciones.
- **Dispersión:** qué tan diferentes son los valores entre las muestras.
- **Tendencia central:** cómo se relacionan la media y la mediana.
- **Variabilidad:** qué tan amplio es el rango intercuartílico.

Por su parte, las barras apiladas permiten comparar la composición de las tres regiones de cada máscara, mientras que la gráfica de pastel resume la distribución global de píxeles del dataset.

> [!IMPORTANT]
> **No debemos confundir la distribución de las características de las imágenes con la distribución global de las clases de segmentación.**
>
> Una gráfica que asigna una observación a cada imagen describe la variabilidad entre muestras. En cambio, una gráfica construida a partir del total de píxeles describe la participación global de cada clase.

---

#### 🤔 Reflexiona

Si tuvieras que preparar este dataset para entrenar una U-Net:

1. ¿Qué diferencias encuentras entre la distribución del número absoluto de píxeles del animal y la distribución de su porcentaje?
2. ¿Qué información adicional aportan las gráficas de violín respecto al histograma?
3. ¿Qué podría indicar una diferencia considerable entre la media y la mediana?
4. ¿Por qué es importante analizar tanto la variabilidad entre imágenes como la distribución global de las clases?
5. ¿Qué posibles dificultades anticiparías a partir de las distribuciones observadas?
6. ¿Qué información conservarías en el manifest para facilitar futuros análisis?

> [!TIP]
> El análisis exploratorio no consiste únicamente en generar gráficas. Su propósito es **transformar las observaciones sobre los datos en información que permita tomar decisiones fundamentadas** durante las siguientes etapas del proyecto.